In [ ]:
%pip install openai pandas tqdm

In [1]:
import os
import json
import re
import time
import pandas as pd
from openai import OpenAI
from dotenv import load_dotenv


load_dotenv()

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key="REDACTED-SEE-.env-AT-REPO-ROOT",
)

In [2]:
# CLASSLA-PIQA datasets
# Place the corresponding *.tsv files in the datasets/ folder
# e.g. datasets/piqa-en.tsv, datasets/piqa-sl.tsv, etc.
# Expected TSV columns: prompt, sol1, sol2, label
tests = [
    "ckm_latin",
    "eng_latin",
    "hrv_latin",
    "mkd_cyrl",
    "slv_latin_cerk",
    "slv_latin",
    "srp_cyrl",
    "srp_latin",
    "srp_tor_cyrl",
    "srp_tor_latin",
    "sl_prl"
]

# OpenRouter models
models = [
    "anthropic/claude-sonnet-4.6",
    "anthropic/claude-opus-4.6",
    "google/gemini-3.1-pro-preview",
    "google/gemini-3.1-flash-lite-preview",
    "openai/gpt-5.4-pro",
    "openai/gpt-5.4",
    "mistralai/mistral-large-2512",
    "mistralai/mistral-small-2603",
    "meta-llama/llama-4-maverick"
]

In [3]:
def predict_gpt(df_test_name, gpt_model):
    os.makedirs("submissions", exist_ok=True)
    model_name = gpt_model.split("/")[1]
    out_path = f"submissions/submission-{model_name}-{df_test_name}.json"

    if os.path.exists(out_path):
        print(f"Skipping (already exists): {out_path}")
        return

    tsv_path = f"datasets/{df_test_name}.tsv"
    if not os.path.exists(tsv_path):
        print(f"WARNING: Missing {tsv_path}, skipping.")
        return
    df = pd.read_csv(tsv_path, sep="\t")

    responses = []
    start_time = time.time()

    for _, entry in df.iterrows():
        prompt = (
            f"### Task\n"
            f"    Given the following situation, which option is more likely to be correct?\n\n"
            f"    Situation: {entry['prompt']}\n\n"
            f"    Option 0: {entry['solution0']}\n\n"
            f"    Option 1: {entry['solution1']}\n\n"
            f"### Output format\n"
            f"    Return a valid JSON dictionary with the following key: 'answer' "
            f"and a value should be either 0 (if option 0 is more plausible) "
            f"or 1 (if option 1 is more plausible). "
            f"Answer ONLY with the JSON dictionary, no explanation."
        )

        completion = None
        for attempt in range(4):
            try:
                completion = client.chat.completions.create(
                    model=gpt_model,
                    response_format={"type": "json_object"},
                    messages=[{"role": "user", "content": prompt}],
                    temperature=0,
                    extra_body={
                                "provider": {
                                    "allow_fallbacks": False, 
                                    "order": ["DeepInfra", "Parasail", "SambaNova", "NovitaAI"],
                                }
                            },
                )
                break
            except Exception as e:
                if "429" in str(e) and attempt < 3:
                    wait = [30, 60, 90][attempt]
                    print(f"Rate limited, waiting {wait}s... (attempt {attempt+1}/4)")
                    time.sleep(wait)
                else:
                    print(f"API error, recording sentinel: {e}")
                    break

        if completion is None:
            responses.append(2)
            continue

        try:
            content = completion.choices[0].message.content
            if content is None:
                responses.append(2)
                continue
            raw = content.strip()
            if raw.startswith("```"):
                raw = raw.split("```")[1]
                if raw.startswith("json"):
                    raw = raw[4:]
                raw = raw.strip()
            if not raw.startswith("{"):
                match = re.search(r'\{[^}]+\}', raw)
                raw = match.group() if match else "{}"
            if not raw.endswith("}"):
                raw += "}"
            response_dict = json.loads(raw)
            predicted = int(response_dict["answer"])
            responses.append(predicted)
        except Exception as e:
            print(f"Error extracting label: {e}")
            responses.append(2)

    elapsed = time.time() - start_time
    n = len(responses)
    print(f"Done. {elapsed/60:.2f} min | {elapsed/n:.3f} s/instance" if n > 0 else f"Done. {elapsed/60:.2f} min | 0 instances")

    with open(out_path, "w") as f:
        json.dump({"system": gpt_model, "predictions": [{"train": "NA (zero-shot)", "test": df_test_name, "predictions": responses}]}, f)
    print(f"Saved: {out_path}")

In [4]:
# Run all models on all datasets
# Tip: comment out models or datasets you want to skip / re-run individually
for model in models:
    for test in tests:
        print(f"\n=== {model} | {test} ===")
        predict_gpt(test, model)


=== anthropic/claude-sonnet-4.6 | ckm_latin ===
Skipping (already exists): submissions/submission-claude-sonnet-4.6-ckm_latin.json

=== anthropic/claude-sonnet-4.6 | eng_latin ===
Skipping (already exists): submissions/submission-claude-sonnet-4.6-eng_latin.json

=== anthropic/claude-sonnet-4.6 | hrv_latin ===
Skipping (already exists): submissions/submission-claude-sonnet-4.6-hrv_latin.json

=== anthropic/claude-sonnet-4.6 | mkd_cyrl ===
Skipping (already exists): submissions/submission-claude-sonnet-4.6-mkd_cyrl.json

=== anthropic/claude-sonnet-4.6 | slv_latin_cerk ===
Skipping (already exists): submissions/submission-claude-sonnet-4.6-slv_latin_cerk.json

=== anthropic/claude-sonnet-4.6 | slv_latin ===
Skipping (already exists): submissions/submission-claude-sonnet-4.6-slv_latin.json

=== anthropic/claude-sonnet-4.6 | srp_cyrl ===
Skipping (already exists): submissions/submission-claude-sonnet-4.6-srp_cyrl.json

=== anthropic/claude-sonnet-4.6 | srp_latin ===
Skipping (already exi

KeyboardInterrupt: 